<a href="https://colab.research.google.com/github/aparna-2001/GATE_PPI/blob/main/mapping_ID_human.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime
import os

os.makedirs('data/raw', exist_ok=True)

In [ ]:
pairs_combined = pd.read_parquet('/content/pairs_combined.parquet')

print(f"Total pairs: {len(pairs_combined):,}")
print(f"\nLabel distribution:")
print(pairs_combined['label'].value_counts())
print(f"\nSource distribution:")
print(pairs_combined['source'].value_counts())

Total pairs: 36,890

Label distribution:
label
0    26485
1    10405
Name: count, dtype: int64

Source distribution:
source
random_negative                      25079
STRING_experimental                  10405
Negatome_combined_stringent_human     1406
Name: count, dtype: int64


In [ ]:
# Re-download if not already present
if not os.path.exists('data/raw/9606.protein.aliases.v12.0.txt.gz'):
    !wget -O data/raw/9606.protein.aliases.v12.0.txt.gz \
      "https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz"

aliases = pd.read_csv(
    'data/raw/9606.protein.aliases.v12.0.txt.gz',
    sep='\t',
    compression='gzip'
)

uniprot_map = aliases[aliases['source'] == 'UniProt_AC'][
    ['#string_protein_id', 'alias']
].copy()
uniprot_map.columns = ['string_id', 'uniprot_id']

uniprot_map['is_swissprot'] = uniprot_map['uniprot_id'].str[0].isin(['P', 'Q', 'O'])

uniprot_map_sorted = uniprot_map.sort_values(
    ['string_id', 'is_swissprot'],
    ascending=[True, False]
)

uniprot_map_final = uniprot_map_sorted.drop_duplicates(
    subset='string_id',
    keep='first'
).reset_index(drop=True)

mapping_dict = dict(zip(uniprot_map_final['string_id'], uniprot_map_final['uniprot_id']))

print(f"Mapping dictionary built: {len(mapping_dict):,} entries")
print(f"Swiss-Prot entries: {uniprot_map_final['is_swissprot'].sum():,}")
print(f"TrEMBL fallbacks:   {(~uniprot_map_final['is_swissprot']).sum():,}")

--2026-06-17 09:53:37--  https://stringdb-downloads.org/download/protein.aliases.v12.0/9606.protein.aliases.v12.0.txt.gz
Resolving stringdb-downloads.org (stringdb-downloads.org)... 49.12.123.75
Connecting to stringdb-downloads.org (stringdb-downloads.org)|49.12.123.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19777800 (19M) [application/octet-stream]
Saving to: ‘data/raw/9606.protein.aliases.v12.0.txt.gz’

data/raw/9606.prote 100%[===================>]  18.86M  16.3MB/s    in 1.2s    

2026-06-17 09:53:39 (16.3 MB/s) - ‘data/raw/9606.protein.aliases.v12.0.txt.gz’ saved [19777800/19777800]

Mapping dictionary built: 19,399 entries
Swiss-Prot entries: 18,302
TrEMBL fallbacks:   1,097


In [ ]:
# Split BEFORE mapping — Negatome pairs already have UniProt IDs
string_pairs   = pairs_combined[pairs_combined['source'] != 'Negatome_combined_stringent_human'].copy()
negatome_pairs = pairs_combined[pairs_combined['source'] == 'Negatome_combined_stringent_human'].copy()

print(f"STRING + random pairs (need mapping): {len(string_pairs):,}")
print(f"Negatome pairs (already UniProt):     {len(negatome_pairs):,}")

# Map ONLY string_pairs
string_pairs['protein1'] = string_pairs['protein1'].map(mapping_dict)
string_pairs['protein2'] = string_pairs['protein2'].map(mapping_dict)

failed_p1 = string_pairs['protein1'].isnull().sum()
failed_p2 = string_pairs['protein2'].isnull().sum()
print(f"\nFailed to map protein1: {failed_p1:,}")
print(f"Failed to map protein2: {failed_p2:,}")

STRING + random pairs (need mapping): 35,484
Negatome pairs (already UniProt):     1,406

Failed to map protein1: 119
Failed to map protein2: 944


In [ ]:
# Find all rows where either protein failed to map
failed_mask = string_pairs['protein1'].isnull() | string_pairs['protein2'].isnull()

print(f"Pairs where protein1 failed:          {string_pairs['protein1'].isnull().sum():,}")
print(f"Pairs where protein2 failed:          {string_pairs['protein2'].isnull().sum():,}")
print(f"Pairs where EITHER failed (to drop):  {failed_mask.sum():,}")
print(f"Pairs where BOTH failed:              {(string_pairs['protein1'].isnull() & string_pairs['protein2'].isnull()).sum():,}")

print(f"\nLabel distribution of failed pairs:")
print(string_pairs[failed_mask]['label'].value_counts())
print(f"\nSource distribution of failed pairs:")
print(string_pairs[failed_mask]['source'].value_counts())

Pairs where protein1 failed:          119
Pairs where protein2 failed:          944
Pairs where EITHER failed (to drop):  1,060
Pairs where BOTH failed:              3

Label distribution of failed pairs:
label
0    668
1    392
Name: count, dtype: int64

Source distribution of failed pairs:
source
random_negative        668
STRING_experimental    392
Name: count, dtype: int64


In [ ]:
# Drop failed pairs from string_pairs
string_pairs_clean = string_pairs[~failed_mask].copy()
print(f"String pairs after dropping failures: {len(string_pairs_clean):,}")

# Recombine with negatome pairs (untouched, already UniProt)
pairs_mapped = pd.concat(
    [string_pairs_clean, negatome_pairs],
    ignore_index=True
)

print(f"\nTotal pairs after mapping: {len(pairs_mapped):,}")
print(f"\nLabel distribution:")
print(pairs_mapped['label'].value_counts())
print(f"\nSource distribution:")
print(pairs_mapped['source'].value_counts())
print(f"\nRatio: {(pairs_mapped['label']==0).sum() / (pairs_mapped['label']==1).sum():.4f}")

String pairs after dropping failures: 34,424

Total pairs after mapping: 35,830

Label distribution:
label
0    25817
1    10013
Name: count, dtype: int64

Source distribution:
source
random_negative                      24411
STRING_experimental                  10013
Negatome_combined_stringent_human     1406
Name: count, dtype: int64

Ratio: 2.5783


In [ ]:
import re

# Check 1 — self-pairs
self_pairs = (pairs_mapped['protein1'] == pairs_mapped['protein2']).sum()
print(f"Self-pairs: {self_pairs}")

# Check 2 — invalid UniProt format
uniprot_pattern = re.compile(r'^[A-Z][0-9][A-Z0-9]{3}[0-9]')
invalid_p1 = pairs_mapped['protein1'].apply(lambda x: not bool(uniprot_pattern.match(str(x)))).sum()
invalid_p2 = pairs_mapped['protein2'].apply(lambda x: not bool(uniprot_pattern.match(str(x)))).sum()
print(f"Invalid format protein1: {invalid_p1}")
print(f"Invalid format protein2: {invalid_p2}")

# Check 3 — duplicates
duplicates = pairs_mapped.duplicated(subset=['protein1', 'protein2']).sum()
print(f"Duplicate pairs: {duplicates}")

# Check 4 — nulls
nulls = pairs_mapped.isnull().sum().sum()
print(f"Nulls: {nulls}")

Self-pairs: 66
Invalid format protein1: 0
Invalid format protein2: 0
Duplicate pairs: 78
Nulls: 0


In [ ]:
# Step 1 — Remove self-pairs
before = len(pairs_mapped)
pairs_mapped = pairs_mapped[
    pairs_mapped['protein1'] != pairs_mapped['protein2']
].copy()
print(f"Removed self-pairs: {before - len(pairs_mapped):,}")

# Step 2 — Remove invalid UniProt format (none expected this time, but check anyway)
valid_mask = (
    pairs_mapped['protein1'].apply(lambda x: bool(uniprot_pattern.match(str(x)))) &
    pairs_mapped['protein2'].apply(lambda x: bool(uniprot_pattern.match(str(x))))
)
before = len(pairs_mapped)
pairs_mapped = pairs_mapped[valid_mask].copy()
print(f"Removed invalid UniProt IDs: {before - len(pairs_mapped):,}")

# Step 3 — Remove duplicates (keep first)
before = len(pairs_mapped)
pairs_mapped = pairs_mapped.drop_duplicates(
    subset=['protein1', 'protein2'],
    keep='first'
).reset_index(drop=True)
print(f"Removed duplicates: {before - len(pairs_mapped):,}")

# Final counts
print(f"\nTotal pairs remaining:  {len(pairs_mapped):,}")
print(f"Positives:              {(pairs_mapped['label']==1).sum():,}")
print(f"Negatives:              {(pairs_mapped['label']==0).sum():,}")
print(f"Ratio:                  {(pairs_mapped['label']==0).sum() / (pairs_mapped['label']==1).sum():.4f}")
print(f"\nSource distribution:")
print(pairs_mapped['source'].value_counts())

Removed self-pairs: 66
Removed invalid UniProt IDs: 0
Removed duplicates: 78

Total pairs remaining:  35,686
Positives:              9,944
Negatives:              25,742
Ratio:                  2.5887

Source distribution:
source
random_negative                      24411
STRING_experimental                   9944
Negatome_combined_stringent_human     1331
Name: count, dtype: int64


In [ ]:
print(f"Nulls:        {pairs_mapped.isnull().sum().sum()}")
print(f"Duplicates:   {pairs_mapped.duplicated(subset=['protein1','protein2']).sum()}")
print(f"Self-pairs:   {(pairs_mapped['protein1']==pairs_mapped['protein2']).sum()}")
invalid_p1 = pairs_mapped['protein1'].apply(lambda x: not bool(uniprot_pattern.match(str(x)))).sum()
invalid_p2 = pairs_mapped['protein2'].apply(lambda x: not bool(uniprot_pattern.match(str(x)))).sum()
print(f"Invalid p1:   {invalid_p1}")
print(f"Invalid p2:   {invalid_p2}")

Nulls:        0
Duplicates:   0
Self-pairs:   0
Invalid p1:   0
Invalid p2:   0


In [ ]:
import json
from datetime import datetime

pairs_mapped.to_csv('data/raw/pairs_mapped.tsv', sep='\t', index=False)
pairs_mapped.to_parquet('data/raw/pairs_mapped.parquet', index=False)

metadata = {
    "date_created":             datetime.today().strftime('%Y-%m-%d'),
    "input_pairs_combined":     36890,
    "negatome_split_before_mapping": True,
    "unmappable_dropped":       1060,
    "self_pairs_dropped":       66,
    "invalid_uniprot_dropped":  0,
    "duplicates_dropped":       78,
    "total_pairs":              len(pairs_mapped),
    "positives":                int((pairs_mapped['label']==1).sum()),
    "negatives":                int((pairs_mapped['label']==0).sum()),
    "ratio":                    round((pairs_mapped['label']==0).sum() /
                                      (pairs_mapped['label']==1).sum(), 4),
    "sources": {
        "STRING_experimental":               int((pairs_mapped['source']=='STRING_experimental').sum()),
        "Negatome_combined_stringent_human": int((pairs_mapped['source']=='Negatome_combined_stringent_human').sum()),
        "random_negative":                   int((pairs_mapped['source']=='random_negative').sum())
    },
    "id_space":                 "UniProt accession (all proteins)",
    "swiss_prot_mappings":      18302,
    "trembl_fallbacks":         1097,
    "correction_note": "Rerun after fixing a bug where Negatome pairs (already UniProt) were incorrectly passed through the Ensembl-to-UniProt mapping_dict, causing 100% of Negatome pairs to be falsely dropped as unmappable. This run correctly splits Negatome out before mapping.",
    "checks_passed": {
        "nulls":        0,
        "duplicates":   0,
        "self_pairs":   0,
        "invalid_ids":  0
    }
}

with open('data/raw/pairs_mapped_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved successfully")
print(json.dumps(metadata, indent=2))

Saved successfully
{
  "date_created": "2026-06-17",
  "input_pairs_combined": 36890,
  "negatome_split_before_mapping": true,
  "unmappable_dropped": 1060,
  "self_pairs_dropped": 66,
  "invalid_uniprot_dropped": 0,
  "duplicates_dropped": 78,
  "total_pairs": 35686,
  "positives": 9944,
  "negatives": 25742,
  "ratio": 2.5887,
  "sources": {
    "STRING_experimental": 9944,
    "Negatome_combined_stringent_human": 1331,
    "random_negative": 24411
  },
  "id_space": "UniProt accession (all proteins)",
  "swiss_prot_mappings": 18302,
  "trembl_fallbacks": 1097,
  "correction_note": "Rerun after fixing a bug where Negatome pairs (already UniProt) were incorrectly passed through the Ensembl-to-UniProt mapping_dict, causing 100% of Negatome pairs to be falsely dropped as unmappable. This run correctly splits Negatome out before mapping.",
  "checks_passed": {
    "nulls": 0,
    "duplicates": 0,
    "self_pairs": 0,
    "invalid_ids": 0
  }
}
